# Định Giá Đúng (The Price is Right)

## Lộ trình Tuần 8

Ngày 1: Modal.com và SpecialistAgent  
Ngày 2: RAG, FrontierAgent, Ensemble Agent  
Ngày 3: ScannerAgent, MessengerAgent  
Ngày 4: AutonomousPlannerAgent và DealAgentFramework  
Ngày 5: Chung kết The Price Is Right

## RAG (Retrieval Augmented Generation) dựa trên bộ dữ liệu 800,000 sản phẩm Amazon đã thu thập

#### Với agent thứ 2, chúng ta sẽ nhờ OpenAI ước tính giá của một sản phẩm - và chúng ta sẽ hỗ trợ thêm cho nó.

Chúng ta đã phát hiện ra rằng các LLM thực sự giỏi trong việc này, ngay cả khi dùng "nguyên bản" (out of the box).

Và chúng ta cũng phát hiện ra rằng có thể vượt qua một LLM hàng đầu (frontier LLM) bằng cách fine-tune một LLM mã nguồn mở.

Bây giờ chúng ta sẽ thử kỹ thuật **inference time** (xử lý tại thời điểm suy luận) thay vì huấn luyện lại -- bằng cách sử dụng RAG!

## 📝 Ghi chú tổng quan notebook

### Tóm tắt quy trình của notebook
Notebook này xây dựng một hệ thống ước tính giá sản phẩm bằng kỹ thuật RAG (Retrieval-Augmented Generation): (1) tải bộ dữ liệu sản phẩm Amazon, (2) mã hoá mô tả sản phẩm thành vector bằng SentenceTransformer, (3) lưu vector vào cơ sở dữ liệu vector Chroma, (4) trực quan hoá dữ liệu vector bằng TSNE (2D/3D), (5) với mỗi sản phẩm cần định giá, tìm các sản phẩm tương tự trong Chroma để làm ngữ cảnh rồi hỏi GPT-5.1 ước tính giá, (6) kết hợp (ensemble) kết quả của RAG-LLM với agent chuyên biệt (Specialist chạy trên Modal) và mạng nơ-ron (Deep Neural Network) để ra giá cuối cùng, (7) đóng gói các bước trên thành các class Agent (FrontierAgent, NeuralNetworkAgent, EnsembleAgent) để tái sử dụng.

### Ý nghĩa chính của notebook
Notebook giải quyết bài toán: cho một mô tả sản phẩm, dự đoán giá bán hợp lý. Dữ liệu đi qua các bước: văn bản thô → vector embedding → tìm kiếm tương tự (similarity search) → tạo ngữ cảnh cho LLM → LLM ước tính giá → kết hợp với các mô hình khác để tăng độ chính xác. Kết quả cuối cùng là các "Agent" có thể tái sử dụng để định giá bất kỳ sản phẩm nào trong toàn bộ Deal Agent Framework của project.

### Mục tiêu cuối cùng
Sau khi chạy và hiểu toàn bộ notebook, ta có được: một Chroma vector store chứa embedding của hàng trăm nghìn sản phẩm; một Agent (FrontierAgent) dùng RAG + GPT-5.1 để định giá; và một Agent tổng hợp (EnsembleAgent) kết hợp nhiều mô hình để cho ra dự đoán giá chính xác hơn - đây là nền tảng cho AutonomousPlannerAgent ở các ngày tiếp theo của Tuần 8.

In [1]:
# Cell này dùng để import toàn bộ thư viện cần thiết cho notebook:
# os/logging để thao tác hệ thống & ghi log, dotenv để đọc file .env,
# huggingface_hub để đăng nhập HuggingFace, numpy/re để xử lý số liệu và chuỗi,
# sentence_transformers để mã hoá văn bản thành vector, chromadb làm vector database,
# sklearn.manifold.TSNE và plotly để trực quan hoá, litellm để gọi LLM,
# tqdm để hiện thanh tiến trình, và các module nội bộ agents.* của project.

import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

In [2]:
# Cell này nạp các biến môi trường từ file .env (ví dụ API key) và
# khai báo tên thư mục DB sẽ dùng để lưu trữ Chroma vector store trên đĩa.

load_dotenv(override=True)
DB = "products_vectorstore"

In [3]:
# Đăng nhập vào HuggingFace bằng token.
# Nếu chưa có tài khoản HuggingFace, có thể đăng ký miễn phí tại www.huggingface.co
# Sau đó thêm HF_TOKEN vào file .env như hướng dẫn trong README của project.
# Cell này cần thiết vì bước tải dataset "ed-donner/items_full" ở cell tiếp theo
# yêu cầu phải xác thực với HuggingFace.

hf_token = os.environ['HF_TOKEN']
login(token=hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# LITE_MODE = True sẽ dùng bộ dữ liệu rút gọn (items_lite) để chạy nhanh hơn,
# còn False sẽ dùng bộ dữ liệu đầy đủ (items_full) với 800,000 sản phẩm.
LITE_MODE = False

In [5]:
# Cell này tải dữ liệu sản phẩm từ HuggingFace Hub (dataset của ed-donner),
# chia sẵn thành 3 tập: train (huấn luyện), val (kiểm định) và test (kiểm thử).
# Tập train sẽ được dùng để nạp vào Chroma vector store ở các cell tiếp theo.

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


# Bây giờ tạo một Chroma Datastore

Chúng ta sẽ sử dụng Chroma - cơ sở dữ liệu vector (Vector Database) miễn phí, mã nguồn mở.  
Chúng ta sẽ tạo một Chroma datastore chứa dữ liệu embedding của các sản phẩm trong tập huấn luyện.

In [6]:
# Khởi tạo client Chroma, lưu dữ liệu bền vững (persistent) trên đĩa tại thư mục DB
# thay vì chỉ lưu tạm trong bộ nhớ - nhờ vậy dữ liệu vẫn còn sau khi tắt notebook.
client = chromadb.PersistentClient(path=DB)

# Giới thiệu mô hình mã hoá (Encoding LLM) SentenceTransformer

all-MiniLM là một mô hình rất hữu ích từ HuggingFace, giúp chuyển câu & đoạn văn thành vector 384 chiều, rất phù hợp cho các tác vụ như tìm kiếm ngữ nghĩa (semantic search).

https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Mô hình này có thể chạy khá nhanh ở máy local.

Ngoài ra, OpenAI cũng cung cấp mô hình Embeddings mã nguồn đóng. So với embeddings của OpenAI, mô hình local có các lợi ích:
1. Miễn phí và nhanh!
2. Chạy được cục bộ (local), nên dữ liệu không bao giờ rời khỏi máy của bạn - hữu ích nếu bạn đang xây dựng một hệ thống RAG cá nhân.

In [7]:
# Tải mô hình mã hoá all-MiniLM-L6-v2 về máy, dùng để chuyển văn bản mô tả sản phẩm
# thành vector 384 chiều phục vụ cho việc lưu trữ và tìm kiếm trong Chroma.
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\user\Desktop\llm_engineering_CuongPhan\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# Truyền vào một danh sách văn bản (list of texts), nhận về một mảng numpy các vector.
# Đây là ví dụ nhanh để xem thử encoder hoạt động ra sao và vector có bao nhiêu chiều (384).

vector = encoder.encode(["A proficient AI engineer who has almost reached the finale of AI Engineering Core Track!"])[0]
print(vector.shape)
vector

(384,)


array([-5.68233915e-02, -6.70465827e-02,  4.41129506e-02,  5.98607305e-03,
       -2.28948668e-02, -2.95300353e-02,  5.56368791e-02,  3.42665240e-02,
       -1.08529389e-01, -3.81690413e-02, -7.43872598e-02, -1.03664272e-01,
        1.69147812e-02,  1.33929204e-03, -6.86191246e-02,  8.99353623e-02,
       -1.45186214e-02, -2.43885256e-02,  4.21854435e-03, -9.62992907e-02,
       -2.51799170e-02,  4.60675657e-02,  4.95350361e-03, -3.88680100e-02,
        1.07717060e-03,  6.82337359e-02, -1.13860117e-02, -5.83416522e-02,
       -1.03801200e-02, -1.74953435e-02, -1.86478421e-02,  4.07058327e-03,
        1.59438271e-02,  6.49722666e-02,  3.71175855e-02,  2.78224908e-02,
       -4.41945344e-02, -2.34372523e-02,  9.71036330e-02, -5.06139211e-02,
       -1.93864480e-02, -3.83471511e-02,  4.76066805e-02, -3.36107500e-02,
        5.08287288e-02,  3.57934907e-02,  2.91815470e-03, -1.06529139e-01,
        4.07211930e-02, -5.85441478e-04, -1.05607428e-01, -1.03584342e-01,
        3.71124037e-02, -

## Với nền tảng đó, hãy nạp dữ liệu vào Chroma database

### Bằng cách tính vector cho 800,000 sản phẩm đã thu thập

Việc này mất khoảng 30 phút trên máy của tác giả (chạy bằng GPU) - có thể sẽ lâu hơn với máy của bạn - hãy dùng bộ dữ liệu Lite nếu cần chạy nhanh hơn!

In [9]:
# Kiểm tra xem collection "products" đã tồn tại trong Chroma chưa; nếu chưa thì tạo mới
# và nạp toàn bộ dữ liệu huấn luyện vào theo từng lô 1000 sản phẩm (để tránh quá tải bộ nhớ).
# Với mỗi lô: lấy mô tả sản phẩm (summary), mã hoá thành vector, kèm metadata (category, price),
# rồi thêm vào collection. Nếu collection đã có sẵn, chỉ cần lấy lại (get_or_create_collection).

collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i: i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

  0%|          | 0/800 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Hãy trực quan hoá dữ liệu đã được vector hoá

In [ ]:
# Rất thú vị nếu tăng con số này lên 800_000 để xem toàn bộ dữ liệu được trực quan hoá,
# nhưng gần như lần nào cũng làm treo máy của tác giả - hãy tự chịu rủi ro nếu muốn thử!!
# Giá trị 10_000 là an toàn để chạy trên hầu hết máy tính.

MAXIMUM_DATAPOINTS = 10_000

In [ ]:
# Khai báo danh sách các danh mục sản phẩm (CATEGORIES) và màu sắc tương ứng (COLORS)
# để tô màu các điểm dữ liệu theo danh mục khi vẽ biểu đồ trực quan hoá bên dưới.

CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [ ]:
# Chuẩn bị dữ liệu trước khi vẽ biểu đồ: lấy tối đa MAXIMUM_DATAPOINTS bản ghi
# (kèm embeddings, documents, metadatas) từ Chroma, sau đó tách riêng vector,
# nội dung văn bản, danh mục, và tính màu sắc tương ứng cho từng điểm dữ liệu.

result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [ ]:
# Hãy thử vẽ biểu đồ 2D
# TSNE (t-distributed Stochastic Neighbor Embedding) là một kỹ thuật phổ biến để
# giảm số chiều của dữ liệu (từ 384 chiều xuống còn 2 chiều) mà vẫn giữ được
# cấu trúc tương đồng giữa các điểm dữ liệu, giúp dễ vẽ và quan sát hơn.

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Vẽ biểu đồ phân tán (scatter plot) 2D: mỗi điểm là một sản phẩm, màu sắc thể hiện
# danh mục sản phẩm, di chuột vào điểm sẽ hiện tên danh mục và một phần mô tả sản phẩm.
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Hãy thử vẽ biểu đồ 3D!
# Lần này giảm dữ liệu xuống còn 3 chiều thay vì 2 chiều để có thêm góc nhìn không gian.

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Vẽ biểu đồ phân tán 3D tương tự bản 2D, nhưng có thêm trục z để quan sát dữ liệu
# theo không gian 3 chiều, giúp thấy rõ hơn các cụm (cluster) sản phẩm theo danh mục.
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Xem thử nội dung của sản phẩm đầu tiên trong tập test - đây sẽ là sản phẩm
# chúng ta dùng làm ví dụ để thử nghiệm việc ước tính giá ở các cell tiếp theo.
test[0]

In [ ]:
# Hàm tiện ích: chuyển mô tả (summary) của một sản phẩm (item) thành vector embedding,
# dùng chung encoder đã tải ở trên.
def vector(item):
    return encoder.encode(item.summary)

In [ ]:
# Đây chính là bước "Retrieval" trong RAG: mã hoá sản phẩm cần định giá thành vector,
# rồi tìm 5 sản phẩm có vector gần giống nhất trong Chroma (query theo embedding).
# Trả về nội dung mô tả và giá của các sản phẩm tương tự để làm ngữ cảnh cho LLM.
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [ ]:
# Thử tìm các sản phẩm tương tự với sản phẩm test[0] để kiểm tra hàm hoạt động đúng.
find_similars(test[0])

In [ ]:
# Chúng ta cần cung cấp ngữ cảnh (context) cho GPT-5.1 bằng cách chọn ra 5 sản phẩm
# có mô tả tương tự - đây là bước "Augmented" trong RAG: bổ sung thông tin tham khảo
# (giá của các sản phẩm tương tự) trước khi hỏi LLM ước tính giá sản phẩm mới.

def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [ ]:
# In thử đoạn ngữ cảnh (context) được tạo ra cho sản phẩm test[0] để kiểm tra nội dung.
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

In [ ]:
# Ghép mô tả sản phẩm cần định giá cùng với đoạn ngữ cảnh (context) ở trên
# thành danh sách messages theo định dạng chat, sẵn sàng để gửi cho LLM (GPT-5.1).
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [ ]:
# In thử nội dung tin nhắn (message) đầy đủ sẽ được gửi cho LLM, để kiểm tra prompt.
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

In [ ]:
# Hàm tổng hợp toàn bộ pipeline RAG: tìm sản phẩm tương tự (Retrieval),
# tạo prompt kèm ngữ cảnh (Augmented), rồi gọi mô hình GPT-5.1 để ước tính giá
# (Generation). seed=42 giúp kết quả ổn định, dễ lặp lại khi kiểm thử.

def gpt_5__1_rag(item):
    documents, prices = find_similars(item)
    response = completion(model="gpt-5.1", messages=messages_for(item, documents, prices), reasoning_effort="none", seed=42)
    return response.choices[0].message.content

In [ ]:
# Xem giá thực tế của sản phẩm test[0] (chiếc pedal distortion) để so sánh
# với giá mà mô hình sẽ ước tính ở cell tiếp theo.

test[0].price

In [ ]:
# Thử nghiệm thực tế: gọi hàm RAG + GPT-5.1 để ước tính giá của test[0].

gpt_5__1_rag(test[0])

In [ ]:
# Đánh giá độ chính xác của hàm gpt_5__1_rag trên toàn bộ tập test,
# so sánh giá dự đoán với giá thực tế để biết mô hình dự đoán tốt đến đâu.
evaluate(gpt_5__1_rag, test)

In [ ]:
# Kết nối tới dịch vụ Pricer đã được deploy trên Modal.com (từ notebook Ngày 1)
# đây chính là "SpecialistAgent" - mô hình fine-tune chuyên định giá sản phẩm.
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [ ]:
# Hàm gọi từ xa (remote call) tới Pricer đang chạy trên Modal để lấy giá dự đoán
# từ mô hình chuyên biệt (specialist model) đã fine-tune.
def specialist(item):
    return pricer.price.remote(item.summary)


In [ ]:
# Hàm tiện ích: trích xuất số tiền (giá) từ câu trả lời dạng text của LLM,
# loại bỏ ký hiệu $ và dấu phẩy, sau đó dùng regex để tìm ra con số đầu tiên.
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

## Tải trọng số (weights) của mạng nơ-ron từ Tuần 6 vào thư mục này

Tệp `deep_neural_network.pth` tại đây:

https://drive.google.com/drive/folders/1uq5C9edPIZ1973dArZiEO-VE13F7m8MK?usp=drive_link

In [ ]:
# Khởi tạo và nạp mô hình mạng nơ-ron (Deep Neural Network) đã huấn luyện từ Tuần 6,
# sau đó định nghĩa hàm deep_neural_network để dùng mô hình này ước tính giá sản phẩm.

from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")

def deep_neural_network(item):
    return runner.inference(item.summary)

In [ ]:
# Hàm Ensemble: kết hợp 3 nguồn dự đoán giá khác nhau (RAG + GPT-5.1, mô hình
# Specialist trên Modal, và mạng nơ-ron) theo trọng số 0.8 / 0.1 / 0.1,
# nhằm tận dụng điểm mạnh của từng mô hình để cho ra kết quả chính xác hơn.
def ensemble(item):
    price1 = get_price(gpt_5__1_rag(item))
    price2 = specialist(item)
    price3 = deep_neural_network(item)
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1


In [ ]:
# Đánh giá hàm ensemble trên tập test để xem việc kết hợp nhiều mô hình
# có cải thiện độ chính xác so với chỉ dùng riêng gpt_5__1_rag hay không.
evaluate(ensemble, test)

In [ ]:
# Bật chế độ ghi log ở mức INFO để có thể theo dõi hoạt động bên trong
# các Agent (FrontierAgent, NeuralNetworkAgent, EnsembleAgent) ở các cell dưới đây.
root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
# Đóng gói toàn bộ pipeline RAG ở trên thành một class FrontierAgent để dễ tái sử dụng.
# Khởi tạo agent với collection Chroma, rồi thử định giá một sản phẩm mới (micro Quadcast).
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price("Quadcast HyperX condenser mic, connects via usb-c to your computer for crystal clear audio")

In [ ]:
# Thử định giá thêm một sản phẩm khác (micro Shure MV7+) bằng FrontierAgent.
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
# Đóng gói mô hình mạng nơ-ron (deep neural network) thành một class NeuralNetworkAgent
# riêng biệt, để có thể dùng độc lập hoặc kết hợp trong EnsembleAgent bên dưới.
from agents.neural_network_agent import NeuralNetworkAgent
agent = NeuralNetworkAgent()


In [ ]:
# Thử định giá micro Shure MV7+ bằng NeuralNetworkAgent để so sánh với FrontierAgent.
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")

In [ ]:
# Đóng gói hàm ensemble thành class EnsembleAgent, kết hợp cả FrontierAgent
# (RAG + GPT-5.1), SpecialistAgent (Modal) và NeuralNetworkAgent thành một Agent duy nhất.
from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)

In [ ]:
# Thử định giá micro Shure MV7+ bằng EnsembleAgent - đây là Agent tổng hợp cuối cùng
# sẽ được dùng trong DealAgentFramework ở các ngày tiếp theo của Tuần 8.
agent.price("Shure MV7+ professional podcaster microphone with usb-c and XLR outputs")